# 08 — Distancia a la línea de pobreza/indigencia (CBA/CBT regional, todos los hogares)

D33 (`docs/decisiones_metodologicas.md`): rediseño del eje del TFI -- el
objeto de estudio pasa a ser la distancia de cada hogar a la línea de
pobreza/indigencia y su evolución, no la condición pobre/no-pobre (que queda
como un derivado). Esta notebook reemplaza a la versión anterior (D25-D27):
cubre **2016T2-2025T4**, con la línea oficial regional de INDEC -- agrega a
cada hogar-trimestre la distancia a la línea de pobreza/indigencia (CBA/CBT
regional por adulto equivalente, todos los hogares, no solo pobres) y valida
contra la tasa oficial de INDEC. La extensión histórica 2003-2006 (isla
`era="historica_gba_proxy"`) va en `08b_retropolacion_pobreza.ipynb`, no acá.

**Lee** `data/02_nucleo/{hogar,individual}`, `data/01_armonizado/hogar` (para
`REGION`, fuera de `ESQUEMA_NUCLEO`, mismo patrón ad hoc que `CAT_INAC`/`CH10`
en la notebook 07). **Escribe** `data/canastas_indec/*_crudo.csv` +
`manifest.json`, `data/02b_distancia_lp/hogar/` (Parquet, reemplaza a
`data/02b_pobreza/hogar/`) y `data/meta/pobreza_validacion_indec.csv` +
`distancia_lp_*.csv`.

**No toca** `src/canasta_regional.py` ni `src/adulto_equivalente.py` (sin
cambios funcionales esta ronda) -- usa `linea_pobreza.calcular_distancia_lp`
(D33/D34, reemplaza a `calcular_linea_pobreza`).

In [ ]:
import sys
import time
from pathlib import Path


def _raiz_repo(desde: Path | None = None) -> Path:
    actual = (desde or Path.cwd()).resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "pytest.ini").exists():
            return candidato
    raise FileNotFoundError(f"no se encontró la raíz del repo subiendo desde {actual}")


REPO = _raiz_repo()
sys.path.insert(0, str(REPO / "src"))

import pandas as pd

import pipeline_meta as pm
from armonizacion import trimestres_publicados
import canasta_regional
import linea_pobreza

TRIMESTRES: list[tuple[int, int]] | None = None
REHACER: bool = False

NUCLEO = REPO / "data" / "02_nucleo"
ARMONIZADO = REPO / "data" / "01_armonizado"
META = REPO / "data" / "meta"
CANASTAS = REPO / "data" / "canastas_indec"
DISTANCIA_LP = REPO / "data" / "02b_distancia_lp" / "hogar"

METODOS_VENTANA = ("a_mismo_trimestre", "b_un_mes_atras")

TODOS_LOS_TRIMESTRES = trimestres_publicados()
trimestres_a_procesar = TRIMESTRES if TRIMESTRES is not None else TODOS_LOS_TRIMESTRES
ES_CORRIDA_PARCIAL = TRIMESTRES is not None
COMMIT_GIT = pm.obtener_commit_git(REPO)
t0_proceso = time.time()
print(f"{len(trimestres_a_procesar)} trimestres de origen a procesar" +
      (" -- CORRIDA PARCIAL (TRIMESTRES acotado)" if ES_CORRIDA_PARCIAL else ""))


## Descarga de CBA/CBT (sección 2 del plan)

Re-ejecuta `canasta_regional.descargar_canastas` contra las dos URLs de
datos.gob.ar (mismo canal que D17, Gini) -- sobreescribe los CSV que ya se
habían bajado ad hoc durante la auditoría previa al plan, para dejar la
descarga reproducible desde la notebook. Si alguna de las dos URLs falla, la
función propaga el error tal cual (D25: escalar, no buscar mirrors).

In [ ]:
manifest_canastas = canasta_regional.descargar_canastas(CANASTAS)
for archivo, info in manifest_canastas["archivos"].items():
    print(f"{archivo}: {info['bytes']:,} bytes")


## Canasta trimestral por región, las dos ventanas (sección 3.A)

`construir_canasta_trimestral` agrega CBA/CBT mensual a trimestral por
`REGION`, una vez por cada `metodo_ventana` -- ninguna de las dos se descarta
todavía, la validación de abajo decide cuál reproduce mejor la tasa oficial.

In [ ]:
canasta_trimestral = {
    metodo: canasta_regional.construir_canasta_trimestral(CANASTAS, metodo)
    for metodo in METODOS_VENTANA
}
for metodo, df in canasta_trimestral.items():
    parciales = df.loc[df["canasta_ventana_parcial"], ["ANO4", "TRIMESTRE"]].drop_duplicates()
    print(f"{metodo}: {len(df)} filas región×trimestre, "
          f"{len(parciales)} trimestre(s) con ventana parcial: "
          f"{sorted(parciales.itertuples(index=False, name=None))}")


## `data/02b_distancia_lp/hogar/` (capa de distancia, D33/D34)

Por cada trimestre 2016T2-2025T4, `construir_filas_distancia` calcula **las
dos** ventanas (`metodo_ventana`) en memoria -- la validación de la sección 4
las usa directamente de ahí, sin releer del disco. A disco se escribe **una
sola vez, filtrado a `metodo_ventana="a_mismo_trimestre"`**, sin las columnas
`metodo_ventana`/`canasta_ventana_parcial` (constantes una vez fijado el
método, D26/D34) y con `era="regional_2016"` (distingue esta capa de la isla
histórica de `08b`, D35). Mismo mecanismo en memoria que evitaba el bug de
reproducibilidad de D27 (la sección 4 siempre tiene las dos ventanas
disponibles, corra la notebook las veces que corra).

In [ ]:
COLUMNAS_HOGAR_NUCLEO = ["CODUSU", "NRO_HOGAR", "ANO4", "TRIMESTRE", "AGLOMERADO", "ITF", "PONDIH",
                          "ingreso_no_declarado"]
COLUMNAS_INDIVIDUAL_NUCLEO = ["CODUSU", "NRO_HOGAR", "COMPONENTE", "CH04", "CH06"]
ERA = "regional_2016"


def _agregar_region(hogar: pd.DataFrame, anio: int, trimestre: int) -> pd.DataFrame:
    p = ARMONIZADO / "hogar" / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet"
    extra = pd.read_parquet(p, columns=["CODUSU", "NRO_HOGAR", "REGION"])
    extra["REGION"] = pd.to_numeric(extra["REGION"], errors="coerce").astype("Int8")
    return hogar.merge(extra, on=["CODUSU", "NRO_HOGAR"], how="left")


def cargar_origen(anio: int, trimestre: int):
    hogar = pm.cargar_hogar(anio, trimestre, COLUMNAS_HOGAR_NUCLEO, NUCLEO)
    individual = pm.cargar_individual(anio, trimestre, COLUMNAS_INDIVIDUAL_NUCLEO, NUCLEO)
    if hogar is None or individual is None:
        return None, None
    return _agregar_region(hogar, anio, trimestre), individual


def construir_filas_distancia(anio: int, trimestre: int) -> pd.DataFrame | None:
    hogar, individual = cargar_origen(anio, trimestre)
    if hogar is None:
        return None
    partes = []
    for metodo in METODOS_VENTANA:
        canasta_trim = canasta_trimestral[metodo]
        canasta_t = canasta_trim[(canasta_trim["ANO4"] == anio) & (canasta_trim["TRIMESTRE"] == trimestre)]
        resultado = linea_pobreza.calcular_distancia_lp(
            hogar, individual, canasta_t[["REGION", "cba_ae", "cbt_ae"]])
        resultado.insert(0, "metodo_ventana", metodo)
        resultado["ANO4"], resultado["TRIMESTRE"], resultado["era"] = anio, trimestre, ERA
        resultado = resultado.merge(canasta_t[["REGION", "canasta_ventana_parcial"]], on="REGION", how="left")
        resultado["canasta_ventana_parcial"] = resultado["canasta_ventana_parcial"].astype("boolean")
        resultado = resultado.merge(hogar[["CODUSU", "NRO_HOGAR", "PONDIH"]], on=["CODUSU", "NRO_HOGAR"], how="left")
        partes.append(resultado)
    return pd.concat(partes, ignore_index=True)


In [ ]:
DISTANCIA_LP.mkdir(parents=True, exist_ok=True)
TRIMESTRES_DISTANCIA = [(a, t) for a, t in trimestres_a_procesar if (a, t) >= (2016, 2)]
METODO_FINAL = "a_mismo_trimestre"
COLUMNAS_A_DESCARTAR_EN_DISCO = ["metodo_ventana", "canasta_ventana_parcial"]

# No reusa particiones existentes (REHACER no aplica acá, a diferencia del
# resto del pipeline): la sección 4 necesita las dos ventanas en memoria en
# cada corrida, así que siempre se recalcula -- escribir a disco es la parte
# barata, filtrada a METODO_FINAL antes de guardar (D27).
filas_por_trimestre: dict[tuple[int, int], pd.DataFrame] = {}
t0_loop = time.time()
for anio, trimestre in TRIMESTRES_DISTANCIA:
    filas = construir_filas_distancia(anio, trimestre)
    if filas is None:
        continue
    filas_por_trimestre[(anio, trimestre)] = filas
    partition_path = DISTANCIA_LP / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet"
    partition_path.parent.mkdir(parents=True, exist_ok=True)
    final = filas[filas["metodo_ventana"] == METODO_FINAL].drop(columns=COLUMNAS_A_DESCARTAR_EN_DISCO)
    final.reset_index(drop=True).to_parquet(partition_path, index=False)

duracion_loop = time.time() - t0_loop
n_filas_disco = sum(len(df[df["metodo_ventana"] == METODO_FINAL]) for df in filas_por_trimestre.values())
print(f"{len(filas_por_trimestre)} trimestres procesados, {n_filas_disco:,} filas en disco "
      f"(solo {METODO_FINAL}), {duracion_loop:.1f}s")

pm.registrar_corrida(
    META / "corridas.csv", "08_distancia_linea_pobreza",
    {"metodo_ventana_final": METODO_FINAL, "TRIMESTRES": trimestres_a_procesar if ES_CORRIDA_PARCIAL else None},
    len(TRIMESTRES_DISTANCIA), n_filas_disco, duracion_loop, COMMIT_GIT,
)


## Sección 4 — Validación contra la tasa oficial de INDEC

Reproduce la tasa de pobreza/indigencia (hogares y personas, ponderado por
`PONDIH` -- hogares directo, personas por `PONDIH × n_miembros`, mismo
criterio que D17) para los 3 semestres acordados: **2S-2016** (arranque de la
serie regional + borde de la ventana "b"), **2S-2024** y **1S-2025** (los más
recientes). Cada semestre oficial agrupa los microdatos de los dos trimestres
que lo componen (así publica INDEC desde que reanudó la serie en 2016 -- un
semestre = la unión de los dos trimestres, no el promedio de dos tasas
trimestrales).

**Valores oficiales, verificados por lectura directa del PDF** (no de un
resumen de búsqueda -- dos veces en esta misma tarea un resumen automático dio
un número equivocado, ver D25): `eph_pobreza_02_16.pdf` (INDEC, *Incidencia de
la pobreza y la indigencia en 31 aglomerados urbanos. Segundo semestre de
2016*, Informes Técnicos Vol. 1 n° 53, 28/3/2017, Cuadro 1) y los informes
`condiciones_de_vida` de 2° semestre de 2024 (Vol. 9 n° 75, 31/3/2025, Cuadro
1) y 1° semestre de 2025 (Vol. 9 n° 237, 25/9/2025, Cuadro 1).

**Umbral de desvío** (D25, mismo estándar que D17/Gini): ≤0,3 pp = aceptable;
0,3-1 pp = revisar; >1 pp = frenar.

In [ ]:
OFICIAL = {
    (2016, 2): {"pobreza_hogares": 21.5, "pobreza_personas": 30.3,
                "indigencia_hogares": 4.5, "indigencia_personas": 6.1},
    (2024, 2): {"pobreza_hogares": 28.6, "pobreza_personas": 38.1,
                "indigencia_hogares": 6.4, "indigencia_personas": 8.2},
    (2025, 1): {"pobreza_hogares": 24.1, "pobreza_personas": 31.6,
                "indigencia_hogares": 5.6, "indigencia_personas": 6.9},
}
SEMESTRES = [(2016, 2, "2S-2016"), (2024, 2, "2S-2024"), (2025, 1, "1S-2025")]
UMBRAL_OK, UMBRAL_REVISAR = 0.3, 1.0


def cargar_semestre(anio: int, semestre: int, metodo: str) -> pd.DataFrame | None:
    """Arma el semestre en memoria desde `filas_por_trimestre` (las dos ventanas,
    D27) -- no relee `data/02b_distancia_lp`, que desde la celda de arriba solo
    tiene METODO_FINAL."""
    trimestres = (1, 2) if semestre == 1 else (3, 4)
    partes = []
    for t in trimestres:
        df_t = filas_por_trimestre.get((anio, t))
        if df_t is None:
            return None
        df = df_t[df_t["metodo_ventana"] == metodo].copy()
        pind = NUCLEO / "individual" / f"ANO4={anio}" / f"TRIMESTRE={t}" / "data.parquet"
        n_miembros = (pd.read_parquet(pind, columns=["CODUSU", "NRO_HOGAR"])
                      .groupby(["CODUSU", "NRO_HOGAR"]).size().rename("n_miembros"))
        df = df.merge(n_miembros, on=["CODUSU", "NRO_HOGAR"], how="left")
        partes.append(df)
    return pd.concat(partes, ignore_index=True)


def tasa_ponderada(df: pd.DataFrame, columna: str, peso: str) -> float:
    valido = df[columna].notna() & df[peso].notna()
    num = (df.loc[valido, columna].astype(float) * df.loc[valido, peso]).sum()
    den = df.loc[valido, peso].sum()
    return 100 * num / den


def tasas_semestre(anio: int, semestre: int, metodo: str) -> dict | None:
    df = cargar_semestre(anio, semestre, metodo)
    if df is None:
        return None
    df["peso_personas"] = df["PONDIH"] * df["n_miembros"]
    return {
        "pobreza_hogares": tasa_ponderada(df, "pobre", "PONDIH"),
        "pobreza_personas": tasa_ponderada(df, "pobre", "peso_personas"),
        "indigencia_hogares": tasa_ponderada(df, "indigente", "PONDIH"),
        "indigencia_personas": tasa_ponderada(df, "indigente", "peso_personas"),
    }


In [ ]:
def clasificar_desvio(desvio: float) -> str:
    if abs(desvio) <= UMBRAL_OK:
        return "aceptable"
    if abs(desvio) <= UMBRAL_REVISAR:
        return "revisar"
    return "frenar"


filas_validacion = []
for anio, semestre, etiqueta in SEMESTRES:
    for metodo in METODOS_VENTANA:
        calc = tasas_semestre(anio, semestre, metodo)
        if calc is None:
            print(f"{etiqueta} / {metodo}: sin datos en data/02b_distancia_lp")
            continue
        oficial = OFICIAL[(anio, semestre)]
        for indicador in ("pobreza_hogares", "pobreza_personas", "indigencia_hogares", "indigencia_personas"):
            desvio = calc[indicador] - oficial[indicador]
            filas_validacion.append({
                "semestre": etiqueta, "metodo_ventana": metodo, "indicador": indicador,
                "oficial": oficial[indicador], "calculado": round(calc[indicador], 2),
                "desvio_pp": round(desvio, 2), "clasificacion": clasificar_desvio(desvio),
            })

pobreza_validacion = pd.DataFrame(filas_validacion)
pobreza_validacion.to_csv(META / "pobreza_validacion_indec.csv", index=False)
print(f"{len(pobreza_validacion)} filas -- data/meta/pobreza_validacion_indec.csv")
print(pobreza_validacion["clasificacion"].value_counts())


### Tablas de desvío, (a) vs (b) lado a lado

**Punto de parada acordado**: esto es lo que hay que revisar antes de tocar
la sección 5 (retropolación) -- la tabla decide qué `metodo_ventana` queda.

In [ ]:
for metodo in METODOS_VENTANA:
    print(f"=== método \"{metodo}\" ===")
    tabla = (pobreza_validacion[pobreza_validacion["metodo_ventana"] == metodo]
             .pivot(index="indicador", columns="semestre", values=["oficial", "calculado", "desvio_pp", "clasificacion"])
             .reindex(["pobreza_hogares", "pobreza_personas", "indigencia_hogares", "indigencia_personas"]))
    display(tabla)


## Chequeo regional, bonus (2S-2016)

El informe de 2S-2016 trae tasas por región estadística (Cuadro 2.1) -- los
de 2S-2024/1S-2025 solo mencionan las regiones de máxima/mínima incidencia en
el texto, no una tabla completa, así que el chequeo regional completo queda
limitado a 2S-2016 esta ronda.

In [ ]:
OFICIAL_REGIONAL_2016_2 = {
    1:  {"pobreza_hogares": 20.8, "pobreza_personas": 29.6, "indigencia_hogares": 4.8, "indigencia_personas": 6.4},   # GBA
    42: {"pobreza_hogares": 25.9, "pobreza_personas": 35.7, "indigencia_hogares": 3.3, "indigencia_personas": 4.3},   # Cuyo
    41: {"pobreza_hogares": 23.5, "pobreza_personas": 33.0, "indigencia_hogares": 3.8, "indigencia_personas": 5.1},   # Noreste
    40: {"pobreza_hogares": 24.2, "pobreza_personas": 31.6, "indigencia_hogares": 4.2, "indigencia_personas": 5.3},   # Noroeste
    43: {"pobreza_hogares": 21.3, "pobreza_personas": 30.5, "indigencia_hogares": 4.7, "indigencia_personas": 6.7},   # Pampeana
    44: {"pobreza_hogares": 16.8, "pobreza_personas": 23.7, "indigencia_hogares": 2.8, "indigencia_personas": 3.3},   # Patagónica
}


def tasas_region_semestre(anio: int, semestre: int, metodo: str) -> pd.DataFrame:
    df = cargar_semestre(anio, semestre, metodo)
    df["peso_personas"] = df["PONDIH"] * df["n_miembros"]
    filas = []
    for region, grupo in df.groupby("REGION"):
        filas.append({
            "REGION": region,
            "pobreza_hogares": tasa_ponderada(grupo, "pobre", "PONDIH"),
            "pobreza_personas": tasa_ponderada(grupo, "pobre", "peso_personas"),
            "indigencia_hogares": tasa_ponderada(grupo, "indigente", "PONDIH"),
            "indigencia_personas": tasa_ponderada(grupo, "indigente", "peso_personas"),
        })
    return pd.DataFrame(filas).set_index("REGION")


filas_regional = []
for metodo in METODOS_VENTANA:
    calc_region = tasas_region_semestre(2016, 2, metodo)
    for region, oficial in OFICIAL_REGIONAL_2016_2.items():
        for indicador in ("pobreza_hogares", "pobreza_personas", "indigencia_hogares", "indigencia_personas"):
            calculado = calc_region.loc[region, indicador]
            filas_regional.append({
                "metodo_ventana": metodo, "REGION": region, "indicador": indicador,
                "oficial": oficial[indicador], "calculado": round(calculado, 2),
                "desvio_pp": round(calculado - oficial[indicador], 2),
            })

pobreza_validacion_regional = pd.DataFrame(filas_regional)
pobreza_validacion_regional.to_csv(META / "pobreza_validacion_regional_2016_2.csv", index=False)
for metodo in METODOS_VENTANA:
    print(f"=== regional 2S-2016, método \"{metodo}\" (|desvío| máximo por indicador) ===")
    sub = pobreza_validacion_regional[pobreza_validacion_regional["metodo_ventana"] == metodo]
    display(sub.loc[sub.groupby("indicador")["desvio_pp"].apply(lambda s: s.abs().idxmax())])


---

## `data/02b_distancia_lp/hogar/` ya queda en su versión final

No hace falta una celda de regeneración aparte -- la celda de construcción de
arriba ya escribe solo `metodo_ventana="a_mismo_trimestre"` en disco, sin esa
columna ni `canasta_ventana_parcial` (D27/D34). Lo que sigue (cobertura,
`CH06`, conteos, distribución de `ratio_lp`, `ITF==0` por región) corre sobre
esa capa tal cual quedó escrita.

## Validaciones pendientes de la sección 4

Sobre `data/02b_distancia_lp/hogar/` ya regenerada (una fila por
hogar-trimestre, método final).

### Cobertura: `REGION` nula y pares (REGION, trimestre) sin canasta

In [ ]:
filas_cobertura = []
for anio, trimestre in TRIMESTRES_DISTANCIA:
    p = DISTANCIA_LP / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet"
    df = pd.read_parquet(p, columns=["REGION"])
    filas_cobertura.append({"ANO4": anio, "TRIMESTRE": trimestre, "n_hogares": len(df),
                             "n_region_nula": int(df["REGION"].isna().sum())})
cobertura_region = pd.DataFrame(filas_cobertura)
total_region_nula = cobertura_region["n_region_nula"].sum()
print(f"hogares con REGION nula, total 2016T2-2025T4: {total_region_nula}")
if total_region_nula:
    display(cobertura_region[cobertura_region["n_region_nula"] > 0])

REGIONES_EPH = {1, 40, 41, 42, 43, 44}
canasta_final = canasta_trimestral[METODO_FINAL]
pares_esperados = {(anio, trimestre, region) for anio, trimestre in TRIMESTRES_DISTANCIA for region in REGIONES_EPH}
pares_presentes = set(canasta_final[["ANO4", "TRIMESTRE", "REGION"]].itertuples(index=False, name=None))
pares_faltantes = sorted(pares_esperados - pares_presentes)
print(f"pares (REGION, trimestre) sin canasta, método {METODO_FINAL}: {len(pares_faltantes)}")
if pares_faltantes:
    print(pares_faltantes)


### Distribución de `CH06`: códigos especiales o fuera de rango (además de `-1`)

In [ ]:
ch06_partes = []
for anio, trimestre in TRIMESTRES_DISTANCIA:
    p = NUCLEO / "individual" / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet"
    ch06_partes.append(pd.read_parquet(p, columns=["CH04", "CH06"]))
ch06_todos = pd.concat(ch06_partes, ignore_index=True)

print(f"CH04, valores observados: {sorted(ch06_todos['CH04'].dropna().unique().tolist())}")
print(f"CH06, min={ch06_todos['CH06'].min()}, max={ch06_todos['CH06'].max()}")

fuera_de_rango = ch06_todos[(ch06_todos["CH06"] < -1) | (ch06_todos["CH06"] > 110)]
ch04_invalido = ch06_todos[~ch06_todos["CH04"].isin([1, 2])]
print(f"CH06 fuera de [-1, 110]: {len(fuera_de_rango)}")
print(f"CH04 fuera de {{1, 2}}: {len(ch04_invalido)}")
print("CH06 negativos observados (conteo):")
display(ch06_todos.loc[ch06_todos["CH06"] < 0, "CH06"].value_counts())


In [ ]:
n_hogares_unidades_na = n_hogares_total = 0
for anio, trimestre in TRIMESTRES_DISTANCIA:
    p = DISTANCIA_LP / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet"
    df = pd.read_parquet(p, columns=["unidades_ae"])
    n_hogares_total += len(df)
    n_hogares_unidades_na += int(df["unidades_ae"].isna().sum())
print(f"hogares con unidades_ae NA (algún integrante con CH04/CH06 indeterminable): "
      f"{n_hogares_unidades_na:,} de {n_hogares_total:,} ({100*n_hogares_unidades_na/n_hogares_total:.4f}%)")


### Conteos por trimestre: no respuesta, `ITF==0` declarado, `ratio_lp` nulo

In [ ]:
filas_conteos = []
for anio, trimestre in TRIMESTRES_DISTANCIA:
    p = DISTANCIA_LP / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet"
    df = pd.read_parquet(p, columns=["ingreso_no_declarado", "itf_cero_declarado", "ratio_lp"])
    filas_conteos.append({
        "ANO4": anio, "TRIMESTRE": trimestre, "n_hogares": len(df),
        "n_ingreso_no_declarado": int((df["ingreso_no_declarado"] == True).sum()),
        "n_itf_cero_declarado": int((df["itf_cero_declarado"] == True).sum()),
        "n_ratio_lp_nulo": int(df["ratio_lp"].isna().sum()),
    })
distancia_lp_conteos_trimestre = pd.DataFrame(filas_conteos)
distancia_lp_conteos_trimestre.to_csv(META / "distancia_lp_conteos_trimestre.csv", index=False)
print(f"data/meta/distancia_lp_conteos_trimestre.csv -- {len(distancia_lp_conteos_trimestre)} filas")
display(distancia_lp_conteos_trimestre.describe())
# n_ratio_lp_nulo debería igualar n_ingreso_no_declarado -- es la única causa de nulo (3.B).
desajuste = distancia_lp_conteos_trimestre[
    distancia_lp_conteos_trimestre["n_ratio_lp_nulo"] != distancia_lp_conteos_trimestre["n_ingreso_no_declarado"]]
print(f"trimestres donde n_ratio_lp_nulo != n_ingreso_no_declarado: {len(desajuste)}")
if len(desajuste):
    display(desajuste)


### Distribución de `ratio_lp` por región y año

In [ ]:
ratio_partes = []
for anio, trimestre in TRIMESTRES_DISTANCIA:
    p = DISTANCIA_LP / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet"
    df = pd.read_parquet(p, columns=["REGION", "ratio_lp"])
    df["ANO4"] = anio
    ratio_partes.append(df)
ratio_todos = pd.concat(ratio_partes, ignore_index=True)
ratio_validos = ratio_todos.dropna(subset=["ratio_lp"]).copy()
ratio_validos["ratio_lp"] = ratio_validos["ratio_lp"].astype(float)

n_negativos = int((ratio_validos["ratio_lp"] < 0).sum())
print(f"ratio_lp < 0: {n_negativos} (esperado 0 -- ITF e ingresos de la canasta son no negativos)")

percentiles_anio = ratio_validos.groupby("ANO4")["ratio_lp"].describe(percentiles=[.01, .05, .5, .95, .99])
percentiles_region = ratio_validos.groupby("REGION")["ratio_lp"].describe(percentiles=[.01, .05, .5, .95, .99])
display(percentiles_anio)
display(percentiles_region)

percentiles_anio.assign(nivel="ANO4").reset_index().rename(columns={"ANO4": "grupo"}).to_csv(
    META / "distancia_lp_ratio_lp_distribucion.csv", index=False)
percentiles_region.assign(nivel="REGION").reset_index().rename(columns={"REGION": "grupo"}).to_csv(
    META / "distancia_lp_ratio_lp_distribucion.csv", mode="a", index=False, header=False)
print("data/meta/distancia_lp_ratio_lp_distribucion.csv escrito (ANO4 + REGION apilados)")


In [ ]:
cerca_de_uno = ratio_validos[(ratio_validos["ratio_lp"] > 0.95) & (ratio_validos["ratio_lp"] < 1.05)]
print(f"hogares con ratio_lp en (0.95, 1.05): {len(cerca_de_uno):,} de {len(ratio_validos):,} "
      f"({100*len(cerca_de_uno)/len(ratio_validos):.2f}%)")
print("conteo por bin de 0.01 alrededor de 1 (busca un pico puntual, no solo densidad creciente):")
bins_cerca_de_uno = ratio_validos["ratio_lp"].round(2).value_counts().sort_index()
display(bins_cerca_de_uno.loc[0.95:1.05])

print()
print("10 valores más altos de ratio_lp:")
display(ratio_validos.nlargest(10, "ratio_lp"))


### `ITF==0` declarado, por región y año (y CABA vs. Partidos del GBA)

El percentil 1 de `ratio_lp` en GBA (`REGION=1`) da 0,0 contra 0,10-0,23 en
el resto -- se investiga acá, **sin cambiar el tratamiento** (sigue siendo
caso válido con `itf_cero_declarado`, D25 sección 3.B): solo describir y
registrar en el log si el patrón es parejo en el tiempo/región o se
concentra en algún lado.

In [ ]:
itf_cero_partes = []
for anio, trimestre in TRIMESTRES_DISTANCIA:
    p = DISTANCIA_LP / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet"
    df = pd.read_parquet(p, columns=["REGION", "ingreso_no_declarado", "itf_cero_declarado"])
    df["ANO4"] = anio
    itf_cero_partes.append(df)
itf_cero_todos = pd.concat(itf_cero_partes, ignore_index=True)
declarado = itf_cero_todos[itf_cero_todos["ingreso_no_declarado"] == False]

por_region_anio = declarado.groupby(["REGION", "ANO4"]).agg(
    n_con_ingreso_declarado=("itf_cero_declarado", "size"),
    n_itf_cero_declarado=("itf_cero_declarado", "sum"),
).reset_index()
por_region_anio["pct_itf_cero"] = (100 * por_region_anio["n_itf_cero_declarado"]
                                    / por_region_anio["n_con_ingreso_declarado"])
por_region_anio.to_csv(META / "distancia_lp_itf_cero_por_region.csv", index=False)
print("data/meta/distancia_lp_itf_cero_por_region.csv escrito")
display(por_region_anio.pivot(index="ANO4", columns="REGION", values="pct_itf_cero").round(2))


In [ ]:
# AGLOMERADO 32 = Ciudad de Buenos Aires (CABA), 33 = Partidos del GBA --
# convención estándar de la EPH; confirmado acá contra el tamaño de muestra
# relativo (33 triplica a 32, consistente con la población real GBA vs. CABA).
ETIQUETA_AGLOMERADO = {32: "CABA", 33: "Partidos del GBA"}

gba_partes = []
for anio, trimestre in TRIMESTRES_DISTANCIA:
    ph = NUCLEO / "hogar" / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet"
    pp = DISTANCIA_LP / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet"
    aglomerado = pd.read_parquet(ph, columns=["CODUSU", "NRO_HOGAR", "AGLOMERADO"])
    df = pd.read_parquet(pp, columns=["CODUSU", "NRO_HOGAR", "REGION", "ingreso_no_declarado", "itf_cero_declarado"])
    df = df[df["REGION"] == 1].merge(aglomerado, on=["CODUSU", "NRO_HOGAR"], how="left")
    df["ANO4"] = anio
    gba_partes.append(df)
gba_todos = pd.concat(gba_partes, ignore_index=True)
gba_todos["zona"] = gba_todos["AGLOMERADO"].map(ETIQUETA_AGLOMERADO)
gba_declarado = gba_todos[gba_todos["ingreso_no_declarado"] == False]

por_zona_anio = gba_declarado.groupby(["zona", "ANO4"]).agg(
    n_con_ingreso_declarado=("itf_cero_declarado", "size"),
    n_itf_cero_declarado=("itf_cero_declarado", "sum"),
).reset_index()
por_zona_anio["pct_itf_cero"] = (100 * por_zona_anio["n_itf_cero_declarado"]
                                  / por_zona_anio["n_con_ingreso_declarado"])
por_zona_anio.to_csv(META / "distancia_lp_itf_cero_caba_gba.csv", index=False)
print("data/meta/distancia_lp_itf_cero_caba_gba.csv escrito")
display(por_zona_anio.pivot(index="ANO4", columns="zona", values="pct_itf_cero").round(2))


---

## Cierre de esta notebook

`data/02b_distancia_lp/hogar/` queda con `era="regional_2016"` en las 666.549
filas de 2016T2-2025T4, con la distancia a la línea (`ratio_lp`,
`brecha_relativa_lp`, `log_ratio_lp` y el mismo trío para indigencia) como
objeto central y `pobre`/`indigente` como derivados (D33). Ver D27 -- fix de
reproducibilidad que sigue vigente sin cambios -- y el resultado de `ITF==0`
por región/año y CABA vs. Partidos del GBA de arriba.

La isla histórica 2003-2006 (`era="historica_gba_proxy"`, condicionada a que
valide contra la tasa oficial de GBA, D35) sigue yendo en
`08b_retropolacion_pobreza.ipynb`, notebook aparte.